# Example 10.2.2 / 1: A lossy delivery arc

Companion notebook for **10.2.2 Lossy and Unit-Converting Networks** in the AMPL lab Documentation, section "A lossy delivery arc". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## A lossy delivery arc

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
param demand >= 0;
param yield_factor > 0, <= 1;
param capacity >= 0;

minimize Sent;

node Source: net_out >= 0;
node Customer: net_in = demand;

arc Shipment >= 0, <= capacity,
    from Source, to Customer yield_factor,
    obj Sent 1;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.param["demand"] = 90
ampl.param["yield_factor"] = 0.9
ampl.param["capacity"] = 120

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.solve_result)
print(ampl.get_value("Shipment"))

Solution determined by presolve;
objective Sent = 100.
solved
100


### Pure AMPL: data.dat and model.run

The same in pure AMPL:

In [6]:
%%writefile data.dat
param demand := 90;
param yield_factor := 0.9;
param capacity := 120;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display Shipment;

Solution determined by presolve;
objective Sent = 100.
Shipment = 100

